# Advanced Clustering Approaches — Worked Examples
**Course:** Data Mining (Professor Nadimi) · **Presenter:** Maryam Mahmoudi

This notebook reproduces every number, table and plot used in the lecture deck `Advanced_Clustering_Approaches.pptx`.
It contains exactly **two worked case studies**, each on one approved real dataset:

| Case | Dataset | Source (DOI) |
|---|---|---|
| 1 | UCI *Wholesale customers* (440 rows) | https://archive.ics.uci.edu/dataset/292/wholesale+customers (10.24432/C5030X, CC BY 4.0) |
| 2 | USGS *High-resolution seismic catalog for the initial 2019 Ridgecrest earthquake sequence* (D. R. Shelly; 34,091 events) | https://www.usgs.gov/data/a-high-resolution-seismic-catalog-initial-2019-ridgecrest-earthquake-sequence (10.5066/P9JN6H0N, CC0) |

**How to run.** Create an environment with the package versions printed in the setup cell, keep the folder layout
(`data/`, `figs/`, `results/` next to this notebook; all paths are relative), then *Kernel → Restart & Run All*.
No network access is needed: the two original source files are stored unchanged in `data/`.

**Provenance rule.** Original files in `data/` are never modified. Derived CSV files carry `source_row_id` (Wholesale: 1-based data-row number in the UCI CSV, header excluded)
or `source_line` / `event_id` (Ridgecrest) and begin with `#` provenance lines (read them with `pd.read_csv(..., comment='#')`).

## Setup / dependencies (exact versions used for the executed run)

```bash
python -m venv .venv && source .venv/bin/activate
pip install numpy==2.5.3 pandas==3.0.5 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 pyproj==3.8.0 jupyter
```
Python 3.13.18. Fonts are optional: plots fall back to DejaVu Sans if *Inter* is not installed. The notebook needs no internet access.

Folder layout (all paths relative): `data/` (original UCI CSV and USGS catalog text file + derived CSVs), `figs/` (PNG plots), `results/results.json` (numbers used on the slides).

In [47]:
import json, itertools, time, platform, importlib.metadata as md
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram, cophenet
from scipy.spatial.distance import pdist, squareform
from scipy.optimize import linear_sum_assignment
from scipy.stats import skew
from sklearn.cluster import KMeans, DBSCAN, OPTICS, HDBSCAN, Birch
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import adjusted_rand_score, silhouette_score
from pyproj import Transformer

DATA, FIG, OUT = Path("data"), Path("figs"), Path("results")
for p in (FIG, OUT): p.mkdir(exist_ok=True)
VERS = {k: md.version(k) for k in ["numpy","pandas","scipy","scikit-learn","matplotlib","pyproj"]}
VERS["python"] = platform.python_version()
print("Package versions used for this run:"); [print(f"  {k}=={v}") for k,v in VERS.items()]

# Palette used on the slides (plots add a few extra categorical hues; marker shapes carry the same information)
INK, BLUE, TEAL, ORANGE, GRID, ICE = "#0F172A", "#0284C7", "#0D9488", "#F97316", "#CBD5E1", "#F8FAFC"
CL = [BLUE, ORANGE, TEAL, "#7C3AED", "#CA8A04", "#BE123C", "#64748B", INK]
MK = ["o", "s", "^", "D", "v", "P", "X", "h"]
plt.rcParams.update({"figure.facecolor": ICE, "axes.facecolor": ICE, "savefig.facecolor": ICE, "text.color": INK,
    "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "axes.edgecolor": INK, "grid.color": GRID,
    "axes.grid": True, "grid.linewidth": .8, "font.family": "DejaVu Sans", "font.size": 13,
    "axes.titlesize": 14, "axes.titleweight": "bold", "legend.frameon": False, "savefig.dpi": 200})
RES = {"versions": VERS}          # everything the slides need is collected here and written to results/results.json
def save(fig, name):
    fig.savefig(FIG / name, bbox_inches="tight"); plt.close(fig); print("saved", FIG / name)
def jsonable(o):
    if isinstance(o, dict): return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [jsonable(v) for v in o]
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (np.bool_,)): return bool(o)
    if isinstance(o, np.ndarray): return jsonable(o.tolist())
    return o
def canon(lab):                       # canonical relabelling by first appearance (for comparing partitions)
    mp = {}; return tuple(mp.setdefault(x, len(mp)) for x in lab)
def prov_csv(df, path, lines):        # write a CSV with '#' provenance header lines
    with open(path, "w") as f:
        for l in lines: f.write("# " + l + "\n")
        df.to_csv(f, index=False)

Package versions used for this run:
  numpy==2.5.3
  pandas==3.0.5
  scipy==1.18.1
  scikit-learn==1.9.1
  matplotlib==3.11.2
  pyproj==3.8.0
  python==3.12.5


# Case 1 — UCI Wholesale customers
**Teaching question.** A wholesale distributor has annual spending data across product categories. Can we identify customer purchasing profiles worth investigating for differentiated account management? Which profiles remain recognizable as we change the linkage or the tree cut?

*Units.* The six spending columns are annual spending in monetary units (m.u.), as stated on the UCI page. `Channel` (1 = Horeca, 2 = Retail) and `Region` (1 = Lisbon, 2 = Oporto, 3 = Other) are **kept out of every distance calculation**; they are used only afterwards to interpret clusters.

In [48]:
w = pd.read_csv(DATA / "Wholesale_customers_data.csv")
w.insert(0, "source_row_id", np.arange(1, len(w) + 1))       # stable identifier = 1-based data-row number in the UCI file
CAT = ["Fresh", "Milk", "Grocery", "Frozen", "Detergents_Paper", "Delicassen"]
assert w.shape == (440, 9) and w.isna().sum().sum() == 0
assert set(w.Channel) == {1, 2} and set(w.Region) == {1, 2, 3}
assert w.Channel.value_counts().to_dict() == {1: 298, 2: 142} and w.Region.value_counts().to_dict() == {3: 316, 1: 77, 2: 47}
print("rows:", len(w), "| missing:", int(w.isna().sum().sum()))
print(w[CAT].mean().round(2).to_dict())
RES["wholesale_source"] = {"rows": 440, "doi": "10.24432/C5030X", "means": w[CAT].mean().round(2).to_dict()}

rows: 440 | missing: 0
{'Fresh': 12000.3, 'Milk': 5796.27, 'Grocery': 7951.28, 'Frozen': 3071.93, 'Detergents_Paper': 2881.49, 'Delicassen': 1524.87}


## 1.1 Feature choice, skew and transformation
Main visual: **Fresh** and **Grocery** (annual spending, m.u.). They are the two largest-volume categories and, as the channel medians below show, they move in *opposite directions* between the two customer types
(Horeca buys more Fresh, Retail more Grocery), so the plane is rich enough to show several regimes without using `Channel` in the distance.
The raw values are strongly right-skewed (a few customers spend 10–100× the median), so Euclidean distance on raw m.u. is dominated by a handful of large accounts.
We use **log10(m.u.)**: equal *ratios* become equal distances (1 unit = a factor of 10). No further z-scoring is applied because both axes are already in the same log10(m.u.) unit; we test the effect of scaling in §1.3.

In [49]:
w["lf"], w["lg"] = np.log10(w.Fresh), np.log10(w.Grocery)       # min(Fresh)=3, min(Grocery)=3 → no zeros, log10 is defined
assert (w[["Fresh", "Grocery"]] > 0).all().all()
sk = {"raw_skew": {c: float(skew(w[c])) for c in ["Fresh","Grocery"]}, "log10_skew": {c: float(skew(w[c2])) for c, c2 in [("Fresh","lf"),("Grocery","lg")]}}
chan_med = w.groupby("Channel")[["Fresh", "Grocery"]].median()
print(sk); print(chan_med)
RES["skew"] = sk; RES["channel_median_fresh_grocery"] = chan_med.to_dict()
fig, ax = plt.subplots(2, 2, figsize=(11, 6.4))
for j, (c, lc) in enumerate([("Fresh", "lf"), ("Grocery", "lg")]):
    ax[0, j].hist(w[c], bins=40, color=BLUE, edgecolor=ICE); ax[0, j].set_title(f"{c}, raw (skew = {sk['raw_skew'][c]:.2f})"); ax[0, j].set_xlabel(f"{c} (annual spending, m.u.)")
    ax[1, j].hist(w[lc], bins=40, color=TEAL, edgecolor=ICE); ax[1, j].set_title(f"log10({c}) (skew = {sk['log10_skew'][c]:.2f})"); ax[1, j].set_xlabel(f"log10 {c} (log10 m.u.)")
    ax[0, j].set_ylabel("customers"); ax[1, j].set_ylabel("customers")
fig.suptitle("All 440 rows (wider view, same two features)", fontsize=12, y=1.0); fig.tight_layout(); save(fig, "w1_skew.png")

{'raw_skew': {'Fresh': 2.5525826879071585, 'Grocery': 3.575187220080786}, 'log10_skew': {'Fresh': -1.6241235263797886, 'Grocery': -0.7371375581266003}}
          Fresh  Grocery
Channel                 
1        9581.5   2684.0
2        5993.5  12390.0
saved figs\w1_skew.png


## 1.2 Reproducible row selection → two nested views of the *same* real customers
* **S40 (analysis subset, n = 40):** `source_row_id % 11 == 2` — a systematic sample of every 11th customer.
* **S8 (hand-trace subset, n = 8):** every 5th row of S40 (positions 0, 5, …, 35).  S8 ⊂ S40 ⊂ all 440 rows, so every hand calculation uses real customers that also appear in the larger analysis.

*How the residue was fixed.* The rule family `source_row_id % 11 == r` (r = 0…10) combined with a start phase p = 0…4 was scanned **in order**, and the first (r, p) meeting three pre-stated teaching criteria was kept:
(i) no tied pairwise distances in S8; (ii) Centroid **and** Median linkage both show a merge-height inversion on S8; (iii) at least 3 different 4-cluster partitions across the six SciPy methods.
This is a *curated teaching sample*, not a random sample; the scan code is below so you can audit it.

In [50]:
def medoid_linkage(X):
    """Classroom medoid-based agglomerative rule (NOT a SciPy built-in):
       cluster representative = medoid = the observed member that minimises the sum of Euclidean distances to the other members (ties -> smallest index);
       merge the pair of clusters whose medoids are closest (ties -> lexicographically smallest pair of cluster ids); recompute the medoid after every merge.
       Returns a SciPy-style linkage matrix [id_i, id_j, medoid_distance, size]."""
    n = len(X); D = squareform(pdist(X)); members = {i: [i] for i in range(n)}; med = {i: i for i in range(n)}; Z = []; nxt = n
    def medoid(m): return m[0] if len(m) == 1 else m[int(np.argmin(D[np.ix_(m, m)].sum(1)))]   # argmin returns the first minimum
    while len(members) > 1:
        best = None
        for a, b in itertools.combinations(sorted(members), 2):
            d = D[med[a], med[b]]
            if best is None or d < best[0] - 1e-12: best = (d, a, b)
        d, a, b = best; m = sorted(members[a] + members[b]); members[nxt] = m; med[nxt] = medoid(m)
        Z.append([a, b, d, len(m)]); del members[a], members[b]; nxt += 1
    return np.array(Z)

def merge_sets(Z, n):                  # list of (set_i, set_j) per merge row
    sets = {i: frozenset([i]) for i in range(n)}; out = []
    for k, (a, b, h, s) in enumerate(Z):
        sets[n + k] = sets[int(a)] | sets[int(b)]; out.append((sets[int(a)], sets[int(b)]))
    return out, sets

METHODS6 = ["single", "complete", "average", "centroid", "median", "ward"]
def all_Z(X): return {**{m: linkage(X, m) for m in METHODS6}, "medoid": medoid_linkage(X)}
def has_inversion(Z): return bool((np.diff(Z[:, 2]) < -1e-12).any())

scan = []
for r in range(11):
    S40_ = w[w.source_row_id % 11 == r].reset_index(drop=True)
    for p in range(5):
        S8_ = S40_.iloc[p::5].reset_index(drop=True)
        if len(S8_) != 8: continue
        X_ = S8_[["lf", "lg"]].values; d_ = pdist(X_)
        no_ties = len(set(np.round(d_, 4))) == len(d_)
        Zs = {m: linkage(X_, m) for m in METHODS6}
        inv = [m for m, Z in Zs.items() if has_inversion(Z)]
        n4 = len({canon(fcluster(Z, 4, "maxclust")) for Z in Zs.values()})
        ok = no_ties and {"centroid", "median"} <= set(inv) and n4 >= 3
        scan.append({"r": r, "phase": p, "no_ties": no_ties, "inversions": inv, "distinct_K4": n4, "meets_criteria": ok})
scan = pd.DataFrame(scan); first = scan[scan.meets_criteria].iloc[0]
print(scan[scan.meets_criteria]); print("first configuration meeting the criteria:", dict(first))
assert (int(first.r), int(first.phase)) == (2, 0)      # regression check: the scan lands on residue 2, phase 0

    r  phase  no_ties          inversions  distinct_K4  meets_criteria
10  2      0     True  [centroid, median]            4            True
first configuration meeting the criteria: {'r': np.int64(2), 'phase': np.int64(0), 'no_ties': np.True_, 'inversions': ['centroid', 'median'], 'distinct_K4': np.int64(4), 'meets_criteria': np.True_}


In [51]:
R_RES, PHASE = int(first.r), int(first.phase)
S40 = w[w.source_row_id % 11 == R_RES].reset_index(drop=True)
S8 = S40.iloc[PHASE::5].reset_index(drop=True)
S8.insert(0, "pt", list("abcdefgh")); S40["in_S8"] = S40.source_row_id.isin(S8.source_row_id)
S40["pt"] = S40.source_row_id.map(dict(zip(S8.source_row_id, S8.pt))).fillna("")
assert len(S40) == 40 and len(S8) == 8
print("S8 source_row_id:", S8.source_row_id.tolist()); print(S8[["pt","source_row_id","Channel","Region","Fresh","Grocery","lf","lg"]].round(3).to_string(index=False))
XS8, XS40 = S8[["lf", "lg"]].values, S40[["lf", "lg"]].values
prov = ["Source: UCI Machine Learning Repository, Wholesale customers (id 292), DOI 10.24432/C5030X, CC BY 4.0; file 'Wholesale customers data.csv' (unchanged copy in data/).",
        "source_row_id = 1-based data-row number in that file (header excluded). Spending columns: annual spending, monetary units (m.u.). lf = log10(Fresh), lg = log10(Grocery).",
        f"Selection: source_row_id % 11 == {R_RES} (S40); S8 = every 5th row of S40 starting at position {PHASE}. Channel/Region are NOT used in distances."]
prov_csv(S8, DATA / "wholesale_S8_hand_trace_8rows.csv", prov + ["S8 = hand-trace subset (n=8), point labels a-h."])
prov_csv(S40.drop(columns=[]), DATA / "wholesale_S40_analysis_40rows.csv", prov + ["S40 = analysis subset (n=40); in_S8 marks the hand-trace rows."])
RES["S8"] = S8.to_dict("records"); RES["S40_ids"] = S40.source_row_id.tolist(); RES["selection_scan"] = scan.to_dict("records"); RES["selection_rule"] = {"residue": R_RES, "phase": PHASE}

fig, ax = plt.subplots(figsize=(7.4, 6))
ax.scatter(w.lf, w.lg, s=14, color=GRID, label="all 440 rows"); ax.scatter(S40.lf, S40.lg, s=34, color=BLUE, label="S40 (n=40)")
ax.scatter(S8.lf, S8.lg, s=120, facecolor="none", edgecolor=ORANGE, linewidth=2, label="S8 (n=8)")
for _, r_ in S8.iterrows(): ax.annotate(r_.pt, (r_.lf, r_.lg), xytext=(7, 6), textcoords="offset points", fontweight="bold")
ax.set_xlabel("log10 Fresh (log10 m.u.)"); ax.set_ylabel("log10 Grocery (log10 m.u.)"); ax.set_title("Nested views of the same customers"); ax.legend(loc="lower right"); save(fig, "w1_subsets.png")

S8 source_row_id: [2, 57, 112, 167, 222, 277, 332, 387]
pt  source_row_id  Channel  Region  Fresh  Grocery    lf    lg
 a              2        2       3   7057     9568 3.849 3.981
 b             57        2       3   4098    26866 3.613 4.429
 c            112        2       3  12579    17569 4.100 4.245
 d            167        2       3   4822     9170 3.683 3.962
 e            222        1       1   5396    10646 3.732 4.027
 f            277        1       3  27901     6964 4.446 3.843
 g            332        2       2  11223    26839 4.050 4.429
 h            387        1       3   1206     2857 3.081 3.456
saved figs\w1_subsets.png


## 1.3 K-means recap on S8 (hand-coded Lloyd iterations) and practical effects on S40
*Objective (plain English):* choose K centres so that the total squared distance from every customer to its own centre is as small as possible.
*Notation:* J(C, μ) = Σₖ Σ_{x∈Cₖ} ‖x − μₖ‖².  Assignment step: nearest centre. Update step: μₖ = mean of the members. J never increases; the loop stops when assignments stop changing.
Teaching input: K = 3 with the initial centroids placed on observations **a, d, h** (chosen so the assignment visibly changes between iteration 1 and 2; other starts are compared in the initialization panel).

In [52]:
def lloyd(X, init_idx, maxit=20):
    c = X[init_idx].copy(); hist = []
    for it in range(maxit):
        d = np.linalg.norm(X[:, None] - c[None], axis=2); lab = d.argmin(1); J = float((d.min(1) ** 2).sum())
        newc = np.array([X[lab == k].mean(0) for k in range(len(c))])
        hist.append({"iter": it + 1, "centroids_in": c.copy(), "labels": lab.copy(), "J": J, "centroids_out": newc.copy(), "dist": d.copy()})
        if np.allclose(newc, c): break
        c = newc
    return hist
INIT = [0, 3, 7]                                                         # a, d, h
H = lloyd(XS8, INIT)
for h in H: print("iter", h["iter"], "labels", "".join(S8.pt[i] + str(l + 1) + " " for i, l in enumerate(h["labels"])), "J=%.4f" % h["J"])
km_sk = KMeans(3, init=XS8[INIT], n_init=1).fit(XS8)
assert canon(km_sk.labels_) == canon(H[-1]["labels"]) and abs(km_sk.inertia_ - H[-1]["J"]) < 1e-9     # hand code == scikit-learn
RES["kmeans_S8"] = [{"iter": h["iter"], "centroids_in": h["centroids_in"], "labels": h["labels"], "J": h["J"], "centroids_out": h["centroids_out"]} for h in H]
RES["kmeans_S8_init"] = [S8.pt[i] for i in INIT]

iter 1 labels a1 b2 c1 d2 e2 f1 g1 h3  J=0.9789
iter 2 labels a2 b2 c1 d2 e2 f1 g1 h3  J=0.4924
iter 3 labels a2 b2 c1 d2 e2 f1 g1 h3  J=0.4488


In [53]:
# Practical effects on S40: (a) choice of K, (b) scaling, (c) initialization, (d) cluster geometry
Ks = range(2, 7); inertia, sil = [], []
for k in Ks:
    km = KMeans(k, n_init=10, random_state=0).fit(XS40); inertia.append(float(km.inertia_)); sil.append(float(silhouette_score(XS40, km.labels_)))
raw40 = S40[["Fresh", "Grocery"]].values
km_log = KMeans(3, n_init=10, random_state=0).fit(XS40); km_raw = KMeans(3, n_init=10, random_state=0).fit(raw40)
ari_scale = adjusted_rand_score(km_log.labels_, km_raw.labels_)
sse_runs, parts = [], set()
for s in range(20):
    km = KMeans(3, init="random", n_init=1, random_state=s).fit(XS40); sse_runs.append(float(km.inertia_)); parts.add(canon(km.labels_))
print("K, inertia, silhouette:", [(k, round(i, 3), round(s_, 3)) for k, i, s_ in zip(Ks, inertia, sil)])
print("ARI(raw-m.u. K-means vs log10 K-means) =", round(ari_scale, 3), "| sizes raw", np.bincount(km_raw.labels_), "log", np.bincount(km_log.labels_))
print("20 random single starts: SSE min/max", round(min(sse_runs), 3), round(max(sse_runs), 3), "| distinct partitions:", len(parts))
RES["kmeans_effects"] = {"K": list(Ks), "inertia": inertia, "silhouette": sil, "ari_raw_vs_log": ari_scale, "sizes_raw": np.bincount(km_raw.labels_), "sizes_log": np.bincount(km_log.labels_),
                         "init_sse_min": min(sse_runs), "init_sse_max": max(sse_runs), "init_distinct_partitions": len(parts)}
fig, ax = plt.subplots(2, 2, figsize=(12, 9.2))
a = ax[0, 0]; a.plot(list(Ks), inertia, "o-", color=BLUE); a.set_xlabel("K"); a.set_ylabel("J (within-cluster SSE, log10 m.u.²)", color=BLUE); a.set_title("(a) Choosing K")
a2 = a.twinx(); a2.plot(list(Ks), sil, "s--", color=ORANGE); a2.set_ylabel("silhouette", color=ORANGE); a2.grid(False)
a = ax[0, 1]
for k in range(3): a.scatter(*XS40[km_raw.labels_ == k].T, s=46, color=CL[k], marker=MK[k])
a.set_title(f"(b) K-means on raw m.u. (shown in log axes)\nARI vs log-space fit = {ari_scale:.2f}"); a.set_xlabel("log10 Fresh"); a.set_ylabel("log10 Grocery")
a = ax[1, 0]; a.hist(sse_runs, bins=12, color=TEAL, edgecolor=ICE); a.axvline(min(sse_runs), color=ORANGE, lw=2); a.set_xlabel("final J of one random start"); a.set_ylabel("runs (of 20)"); a.set_title(f"(c) Initialization: {len(parts)} distinct partitions")
a = ax[1, 1]
gx, gy = np.meshgrid(np.linspace(XS40[:, 0].min() - .3, XS40[:, 0].max() + .3, 300), np.linspace(XS40[:, 1].min() - .3, XS40[:, 1].max() + .3, 300))
zz = km_log.predict(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape); a.contourf(gx, gy, zz, levels=[-.5, .5, 1.5, 2.5], colors=[BLUE, ORANGE, TEAL], alpha=.12)
for k in range(3): a.scatter(*XS40[km_log.labels_ == k].T, s=46, color=CL[k], marker=MK[k])
a.scatter(*km_log.cluster_centers_.T, marker="*", s=300, color=INK, zorder=5); a.set_title("(d) Geometry: straight Voronoi boundaries (K=3)"); a.set_xlabel("log10 Fresh"); a.set_ylabel("log10 Grocery")
fig.tight_layout(); save(fig, "w1_kmeans_effects.png")

K, inertia, silhouette: [(2, 12.419, 0.426), (3, 7.034, 0.409), (4, 5.281, 0.432), (5, 3.612, 0.42), (6, 2.808, 0.413)]
ARI(raw-m.u. K-means vs log10 K-means) = -0.015 | sizes raw [23  8  9] log [21 12  7]
20 random single starts: SSE min/max 7.034 7.511 | distinct partitions: 9
saved figs\w1_kmeans_effects.png


## 1.4 Dissimilarity matrix of S8, AGNES (Average linkage) by hand
Representation used in the worked algorithm: a **dissimilarity (distance)** matrix — Euclidean distance in log10(m.u.) — so the algorithm merges the **smallest** off-diagonal entry.
If we were given a *similarity* s (larger = closer, e.g. s = 1/(1+d) or the negative distance) the selection direction flips: merge the **largest** s. Both pick the same pair when s is a decreasing function of d.
*Tie-breaking rule:* among equal minima take the pair whose first member has the smallest index, then the smallest second index (S8 has no ties, so it is never triggered).

In [54]:
D8 = squareform(pdist(XS8)); names = list(S8.pt)
print(pd.DataFrame(D8, index=names, columns=names).round(3))
def agnes_by_hand(D, method="average"):
    """Explicit AGNES with Lance-Williams updates (single/complete/average). Returns step records incl. the full matrix before each merge."""
    n = len(D); act = [[i] for i in range(n)]; M = D.copy().astype(float); steps = []; nid = n; ids = list(range(n))
    while len(act) > 1:
        best = None
        for i, j in itertools.combinations(range(len(act)), 2):
            if best is None or M[i, j] < best[0] - 1e-12: best = (M[i, j], i, j)
        h, i, j = best; ni, nj = len(act[i]), len(act[j])
        steps.append({"labels": ["".join(names[k] for k in c) for c in act], "matrix": M.copy(), "pair": (i, j), "height": float(h), "members": sorted(act[i] + act[j]), "ids": (ids[i], ids[j], nid)})
        new = []
        for k in range(len(act)):
            if k in (i, j): continue
            if method == "single": v = min(M[i, k], M[j, k])
            elif method == "complete": v = max(M[i, k], M[j, k])
            else: v = (ni * M[i, k] + nj * M[j, k]) / (ni + nj)
            new.append(v)
        keep = [k for k in range(len(act)) if k not in (i, j)]
        M2 = np.zeros((len(keep) + 1, len(keep) + 1)); M2[:-1, :-1] = M[np.ix_(keep, keep)]; M2[-1, :-1] = M2[:-1, -1] = new
        act = [act[k] for k in keep] + [sorted(act[i] + act[j])]; ids = [ids[k] for k in keep] + [nid]; nid += 1; M = M2
    return steps
AG = agnes_by_hand(D8, "average")
Zavg8 = linkage(XS8, "average")
assert np.allclose([s["height"] for s in AG], Zavg8[:, 2]) and all(frozenset(s["members"]) == ms[0] | ms[1] for s, ms in zip(AG, merge_sets(Zavg8, 8)[0]))
for k, s in enumerate(AG[:4]): print(f"merge {k+1}: {s['labels'][s['pair'][0]]} + {s['labels'][s['pair'][1]]}  @ {s['height']:.4f}")
RES["D8"] = D8; RES["agnes_avg_S8"] = [{"labels": s["labels"], "matrix": s["matrix"], "pair": s["pair"], "height": s["height"], "members": [names[i] for i in s["members"]]} for s in AG]

       a      b      c      d      e      f      g      h
a  0.000  0.507  0.364  0.166  0.125  0.613  0.491  0.930
b  0.507  0.000  0.521  0.472  0.419  1.019  0.438  1.109
c  0.364  0.521  0.000  0.503  0.427  0.530  0.191  1.288
d  0.166  0.472  0.503  0.000  0.081  0.772  0.593  0.787
e  0.125  0.419  0.427  0.081  0.000  0.737  0.512  0.866
f  0.613  1.019  0.530  0.772  0.737  0.000  0.707  1.418
g  0.491  0.438  0.191  0.593  0.512  0.707  0.000  1.373
h  0.930  1.109  1.288  0.787  0.866  1.418  1.373  0.000
merge 1: d + e  @ 0.0812
merge 2: a + de  @ 0.1459
merge 3: c + g  @ 0.1906
merge 4: b + ade  @ 0.4661


## 1.5 DIANA by hand (Kaufman & Rousseeuw 1990)
*Rule.* (1) Split the cluster with the **largest diameter** (maximum pairwise dissimilarity). (2) The initial **splinter** is the member with the largest *average* dissimilarity to the other members. (3) Repeatedly move to the splinter group the object i of the remaining group R with the largest positive value of
`mean d(i, R∖{i}) − mean d(i, S)`; stop when no difference is positive. (4) The **split height** is the diameter of the cluster that was split. This differs from a generic *bisecting* divisive method, which would run an arbitrary 2-partitioning (e.g. 2-means) on the chosen cluster and has no diameter-based height.

In [55]:
def diana(D):
    n = len(D); clusters = [list(range(n))]; log = []; nid = 0
    def tree_leaf(i): return {"members": [i]}
    while any(len(c) > 1 for c in clusters):
        diam = [D[np.ix_(c, c)].max() if len(c) > 1 else -1 for c in clusters]; ci = int(np.argmax(diam)); C = clusters[ci]
        avg = {i: float(D[i, [j for j in C if j != i]].mean()) for i in C}; s = max(C, key=lambda i: (avg[i], -i))
        S, Rm, rounds = [s], [j for j in C if j != s], []
        while len(Rm) > 1:
            tab = {i: (float(D[i, [j for j in Rm if j != i]].mean()), float(D[i, S].mean())) for i in Rm}
            diffs = {i: a - b for i, (a, b) in tab.items()}; best = max(Rm, key=lambda i: (diffs[i], -i))
            rounds.append({"table": tab, "diffs": diffs, "move": best if diffs[best] > 0 else None})
            if diffs[best] <= 0: break
            S.append(best); Rm.remove(best)
        log.append({"parent": sorted(C), "diameter": float(D[np.ix_(C, C)].max()), "avg_to_others": avg, "splinter": s, "rounds": rounds, "S": sorted(S), "R": sorted(Rm), "others_diameters": diam})
        clusters[ci:ci + 1] = [sorted(S), sorted(Rm)]
    return log
DL = diana(D8)
for k, s in enumerate(DL): print(f"split {k+1}: {''.join(names[i] for i in s['parent'])} (diameter {s['diameter']:.3f}) -> {''.join(names[i] for i in s['S'])} | {''.join(names[i] for i in s['R'])}")
assert DL[0]["diameter"] == D8.max()
RES["diana_S8"] = [{**s, "avg_to_others": {names[k]: v for k, v in s["avg_to_others"].items()},
    "rounds": [{"table": {names[k]: v for k, v in r["table"].items()}, "diffs": {names[k]: v for k, v in r["diffs"].items()}, "move": None if r["move"] is None else names[r["move"]]} for r in s["rounds"]],
    "parent": [names[i] for i in s["parent"]], "S": [names[i] for i in s["S"]], "R": [names[i] for i in s["R"]], "splinter": names[s["splinter"]]} for s in DL]

split 1: abcdefgh (diameter 1.418) -> h | abcdefg
split 2: abcdefg (diameter 1.019) -> f | abcdeg
split 3: abcdeg (diameter 0.593) -> bcg | ade
split 4: bcg (diameter 0.521) -> b | cg
split 5: cg (diameter 0.191) -> c | g
split 6: ade (diameter 0.166) -> a | de
split 7: de (diameter 0.081) -> d | e


## 1.6 Seven linkage methods on the same S8 observations
All seven start from the **same three merges** (d+e, then a joins {d,e}, then c+g). From that shared 5-cluster state we ask each method one question: *which two clusters merge next?*
The quantity each method minimises: Single = min pair distance · Complete = max pair distance · Average = mean over all cross pairs · Centroid = distance between **all-observation centroids** ·
Median (WPGMC) = distance between recursively maintained representatives, where a merged cluster's representative is the **midpoint of its two parents' representatives** (equal weights, not the coordinate-wise median) ·
Medoid-based (classroom rule) = distance between cluster **medoids** · Ward = increase in within-cluster sum of squares ΔESS = (nᵢnⱼ/(nᵢ+nⱼ))·‖cᵢ−cⱼ‖² (SciPy plots √(2·ΔESS)).

In [56]:
def partition_k(Z, n, k):                       # partition after n-k merges, i.e. by MERGE ORDER (valid even when heights are non-monotone)
    sets = {i: frozenset([i]) for i in range(n)}; cur = set(sets)
    for t, (a, b, h, s) in enumerate(Z[: n - k]):
        sets[n + t] = sets[int(a)] | sets[int(b)]; cur -= {int(a), int(b)}; cur.add(n + t)
    lab = np.zeros(n, int)
    for c, cid in enumerate(sorted(cur, key=lambda c_: min(sets[c_]))):
        for i in sets[cid]: lab[i] = c
    return lab
def cophenetic(Z, n):                           # height of the merge that first joins each pair (works for non-monotone trees)
    C = np.zeros((n, n)); sets = {i: [i] for i in range(n)}
    for t, (a, b, h, s) in enumerate(Z):
        A_, B_ = sets[int(a)], sets[int(b)]
        for i in A_:
            for j in B_: C[i, j] = C[j, i] = h
        sets[n + t] = A_ + B_
    return squareform(C, checks=False)
def reps_wpgmc(Z, X):                           # WPGMC representatives: midpoint of the two parents' representatives
    n = len(X); rep = {i: X[i].copy() for i in range(n)}
    for t, (a, b, h, s) in enumerate(Z): rep[n + t] = (rep[int(a)] + rep[int(b)]) / 2
    return rep

Zs8 = all_Z(XS8); sets8 = {m: merge_sets(Z, 8)[0] for m, Z in Zs8.items()}
first3 = {m: [frozenset(a | b) for a, b in s[:3]] for m, s in sets8.items()}
assert all(first3[m] == first3["average"] for m in Zs8)                      # shared first three merges
lab5 = partition_k(Zs8["average"], 8, 5); STATE = [[i for i in range(8) if lab5[i] == c] for c in range(5)]
print("shared 5-cluster state:", ["".join(names[i] for i in c) for c in STATE])
rep = reps_wpgmc(Zs8["median"], XS8)
def rep_of(c):                                  # WPGMC representative of a cluster of the shared state (its id in the median linkage)
    ids = {frozenset(m): k for k, m in merge_sets(Zs8["median"], 8)[1].items()}; return rep[ids[frozenset(c)]]
def medoid_of(c): return c[0] if len(c) == 1 else c[int(np.argmin(D8[np.ix_(c, c)].sum(1))) ]
def cand_value(method, ci, cj):
    A_, B_ = STATE[ci], STATE[cj]; cross = D8[np.ix_(A_, B_)]
    if method == "single": return float(cross.min())
    if method == "complete": return float(cross.max())
    if method == "average": return float(cross.mean())
    if method == "centroid": return float(np.linalg.norm(XS8[A_].mean(0) - XS8[B_].mean(0)))
    if method == "median": return float(np.linalg.norm(rep_of(A_) - rep_of(B_)))
    if method == "medoid": return float(D8[medoid_of(A_), medoid_of(B_)])
    if method == "ward": return float(len(A_) * len(B_) / (len(A_) + len(B_)) * np.sum((XS8[A_].mean(0) - XS8[B_].mean(0)) ** 2))   # ΔESS
PAIRS = list(itertools.combinations(range(5), 2)); SL = ["".join(names[i] for i in c) for c in STATE]
tab = pd.DataFrame({m: [cand_value(m, i, j) for i, j in PAIRS] for m in ["single", "complete", "average", "centroid", "median", "medoid", "ward"]},
                   index=[f"{SL[i]}+{SL[j]}" for i, j in PAIRS])
tab["ward_height_sqrt2dESS"] = np.sqrt(2 * tab["ward"])
print(tab.round(3)); chosen = tab[["single", "complete", "average", "centroid", "median", "medoid", "ward"]].idxmin().to_dict(); print(chosen)
for m in ["single", "complete", "average", "centroid", "median", "medoid", "ward"]:   # the decision recomputed here must equal the 4th merge in the linkage matrix
    a_, b_ = sets8[m][3]; want = {frozenset(names[i] for i in a_), frozenset(names[i] for i in b_)}; got = {frozenset(x) for x in chosen[m].split("+")}
    assert got == want, (m, chosen[m])
ade = STATE[[k for k, c in enumerate(STATE) if len(c) == 3][0]]
cent_ade, wp_ade, med_stat = XS8[ade].mean(0), rep_of(ade), np.median(XS8[ade], axis=0)
print("{ade}: centroid", cent_ade.round(4), "| WPGMC rep", wp_ade.round(4), "| coordinate-wise median", med_stat.round(4), "| medoid", names[medoid_of(ade)])
RES["shared_state"] = {"clusters": SL, "pairs": [f"{SL[i]}+{SL[j]}" for i, j in PAIRS], "table": tab.to_dict("list"), "chosen": chosen,
                       "ade": {"centroid": cent_ade, "wpgmc": wp_ade, "stat_median": med_stat, "medoid": names[medoid_of(ade)], "members": [names[i] for i in ade]},
                       "wpgmc_reps": {SL[k]: rep_of(c) for k, c in enumerate(STATE)}, "medoids": {SL[k]: names[medoid_of(c)] for k, c in enumerate(STATE)},
                       "centroids": {SL[k]: XS8[c].mean(0) for k, c in enumerate(STATE)}}

shared 5-cluster state: ['ade', 'b', 'cg', 'f', 'h']
        single  complete  average  centroid  median  medoid   ward  \
ade+b    0.419     0.507    0.466     0.461   0.471   0.419  0.160   
ade+cg   0.364     0.593    0.482     0.472   0.458   0.427  0.267   
ade+f    0.613     0.772    0.707     0.706   0.683   0.737  0.374   
ade+h    0.787     0.930    0.861     0.859   0.877   0.866  0.554   
b+cg     0.438     0.521    0.479     0.471   0.471   0.521  0.148   
b+f      1.019     1.019    1.019     1.019   1.019   1.019  0.519   
b+h      1.109     1.109    1.109     1.109   1.109   1.109  0.615   
cg+f     0.530     0.707    0.619     0.618   0.618   0.530  0.254   
cg+h     1.288     1.373    1.331     1.328   1.328   1.288  1.175   
f+h      1.418     1.418    1.418     1.418   1.418   1.418  1.005   

        ward_height_sqrt2dESS  
ade+b                   0.565  
ade+cg                  0.731  
ade+f                   0.865  
ade+h                   1.053  
b+cg            

In [57]:
# Complete merge lists, inversion check and K=4 partitions on S8 (all seven methods, partition by MERGE ORDER)
S8_methods = {}
for m, Z in Zs8.items():
    sets_, _ = merge_sets(Z, 8)
    S8_methods[m] = {"Z": Z, "merges": [["".join(names[i] for i in sorted(a)), "".join(names[i] for i in sorted(b)), float(Z[t, 2])] for t, (a, b) in enumerate(sets_)],
                     "inversion": has_inversion(Z), "K4": [names[i] + str(partition_k(Z, 8, 4)[i]) for i in range(8)], "K3": partition_k(Z, 8, 3), "K4_labels": partition_k(Z, 8, 4)}
    if m in METHODS6 and not has_inversion(Z): assert canon(fcluster(Z, 4, "maxclust")) == canon(partition_k(Z, 8, 4))
inv_info = {}
for m in ("centroid", "median"):
    Z = Zs8[m]; t = int(np.where(np.diff(Z[:, 2]) < -1e-12)[0][0]); inv_info[m] = {"merge_index": t + 1, "h_before": float(Z[t, 2]), "h_after": float(Z[t + 1, 2]), "cut_between": float((Z[t, 2] + Z[t + 1, 2]) / 2)}
print({m: S8_methods[m]["inversion"] for m in S8_methods}); print(inv_info)
for m in S8_methods: print(m.ljust(9), [f"{a}+{b}@{h:.3f}" for a, b, h in S8_methods[m]["merges"]])
RES["S8_methods"] = {m: {k: v for k, v in d.items() if k != "Z"} | {"Z": d["Z"]} for m, d in S8_methods.items()}; RES["inversion_S8"] = inv_info

{'single': False, 'complete': False, 'average': False, 'centroid': True, 'median': True, 'ward': False, 'medoid': False}
{'centroid': {'merge_index': 4, 'h_before': 0.4614890944522541, 'h_after': 0.4273931367394283, 'cut_between': 0.4444411155958412}, 'median': {'merge_index': 4, 'h_before': 0.4580691236167684, 'h_after': 0.4120723345127284, 'cut_between': 0.4350707290647484}}
single    ['d+e@0.081', 'a+de@0.125', 'c+g@0.191', 'ade+cg@0.364', 'b+acdeg@0.419', 'f+abcdeg@0.530', 'h+abcdefg@0.787']
complete  ['d+e@0.081', 'a+de@0.166', 'c+g@0.191', 'b+ade@0.507', 'cg+abde@0.593', 'f+abcdeg@1.019', 'h+abcdefg@1.418']
average   ['d+e@0.081', 'a+de@0.146', 'c+g@0.191', 'b+ade@0.466', 'cg+abde@0.481', 'f+abcdeg@0.730', 'h+abcdefg@1.110']
centroid  ['d+e@0.081', 'a+de@0.142', 'c+g@0.191', 'b+ade@0.461', 'cg+abde@0.427', 'f+abcdeg@0.695', 'h+abcdefg@1.080']
median    ['d+e@0.081', 'a+de@0.142', 'c+g@0.191', 'ade+cg@0.458', 'b+acdeg@0.412', 'f+abcdeg@0.814', 'h+abcdefg@1.196']
ward      ['d+e@0.

## 1.7 UCI walkthrough on S40: distance representation, six SciPy methods, medoid rule, comparison plots
`pdist` returns the condensed vector of the 780 pairwise distances; `linkage(X, method)` with raw observations lets Centroid/Median/Ward use their Euclidean geometry.
**Reading a linkage-matrix row** `[i, j, h, s]`: clusters `i` and `j` are merged at height `h` into a new cluster of size `s`; ids 0…39 are original observations and id `40 + r` is the cluster created by row `r`.

In [58]:
dist40 = pdist(XS40); print("pairwise distances:", dist40.shape, "| max =", dist40.max().round(3))
Zs40 = all_Z(XS40); nL = 40
for m in ["average"]: print(pd.DataFrame(Zs40[m][:4], columns=["i", "j", "height", "size"]).round(3))
rows40 = []
for m, Z in Zs40.items():
    cp = cophenetic(Z, nL); rows40.append({"method": m, "inversion": has_inversion(Z), "cophenetic_corr": float(np.corrcoef(cp, dist40)[0, 1]), "top_height": float(Z[-1, 2]), "n_inversions": int((np.diff(Z[:, 2]) < -1e-12).sum())})
M40 = pd.DataFrame(rows40); print(M40.round(3))
RES["S40_methods"] = M40.to_dict("records"); RES["Z40_average_head"] = Zs40["average"][:4]

# Partitions at a shared cluster count K = 4, labels mapped onto the Ward reference by maximum overlap (Hungarian)
K = 4; P = {m: partition_k(Z, nL, K) for m, Z in Zs40.items()}
def map_to(ref, lab):
    C = np.array([[np.sum((ref == a) & (lab == b)) for b in range(K)] for a in range(K)]); r_, c_ = linear_sum_assignment(-C); mp = {c: r for r, c in zip(r_, c_)}
    return np.array([mp[x] for x in lab])
PM = {m: map_to(P["ward"], lab) for m, lab in P.items()}
ARI = pd.DataFrame([[adjusted_rand_score(P[a], P[b]) for b in Zs40] for a in Zs40], index=list(Zs40), columns=list(Zs40)); print(ARI.round(2))
def jacc(a, b): return len(a & b) / len(a | b)
stab = {m: [max(jacc(set(np.where(PM["ward"] == k)[0]), set(np.where(PM[m] == c)[0])) for c in range(K)) for k in range(K)] for m in Zs40}
sizes = {m: np.bincount(PM[m], minlength=K).tolist() for m in Zs40}; print(pd.DataFrame(stab).round(2)); print(sizes)
RES["partitions_K4"] = {"labels": {m: PM[m] for m in PM}, "sizes": sizes, "ARI": ARI.round(4).to_dict(), "jaccard_vs_ward": stab}
cr = pd.crosstab(PM["ward"], PM["average"], rownames=["Ward"], colnames=["Average"]); print(cr); RES["crosstab_ward_average"] = cr.values.tolist()

TITLE = {"single": "Single", "complete": "Complete", "average": "Average", "centroid": "Centroid", "median": "Median (WPGMC)", "medoid": "Medoid-based", "ward": "Ward"}
YLAB = {m: "height = Euclid. distance (log10 m.u.)" for m in Zs40}; YLAB["ward"] = "height = sqrt(2·ΔESS)"
def draw_dend(m, ax, cut=None, colors=None):
    Z = Zs40[m]
    d = dendrogram(Z, labels=S40.source_row_id.values, ax=ax, color_threshold=0, above_threshold_color=INK, leaf_rotation=90, leaf_font_size=8)
    ax.set_title(TITLE[m] + (" — inversion present" if has_inversion(Z) else "")); ax.set_ylabel(YLAB[m], fontsize=10); ax.grid(axis="x", visible=False)
    return d
for grp, nm in [(["single", "complete", "average", "centroid"], "a"), (["median", "medoid", "ward"], "b")]:
    fig, axs = plt.subplots(2, 2, figsize=(13, 8.4)); axs = axs.ravel()
    for a, m in zip(axs, grp): draw_dend(m, a)
    if len(grp) < 4:
        a = axs[3]; a.axis("off"); a.text(0, .5, "Raw heights are shown on each panel.\nCentroid/Median/Medoid heights can decrease\n(inversion); Ward height = sqrt(2·ΔESS).\nLeaves = source_row_id of the 40 customers.", fontsize=13, va="center")
    fig.tight_layout(); save(fig, f"w1_dendrograms_{nm}.png")
for grp, nm in [(["single", "complete", "average", "centroid"], "a"), (["median", "medoid", "ward"], "b")]:
    fig, axs = plt.subplots(2, 2, figsize=(11.4, 9.4)); axs = axs.ravel()
    for a, m in zip(axs, grp):
        for k in range(K):
            pts = XS40[PM[m] == k]; a.scatter(*pts.T, s=48, color=CL[k], marker=MK[k], label=f"cluster {k+1} (n={len(pts)})")
        a.set_title(f"{TITLE[m]} · K=4"); a.set_xlabel("log10 Fresh (log10 m.u.)"); a.set_ylabel("log10 Grocery (log10 m.u.)")
        a.set_xlim(XS40[:, 0].min() - .2, XS40[:, 0].max() + .2); a.set_ylim(XS40[:, 1].min() - .2, XS40[:, 1].max() + .2); a.legend(fontsize=9, loc="lower right")
    if len(grp) < 4: axs[3].axis("off")
    fig.tight_layout(); save(fig, f"w1_partitions_k4_{nm}.png")

pairwise distances: (780,) | max = 2.798
      i     j  height  size
0   0.0  27.0   0.026   2.0
1  18.0  38.0   0.044   2.0
2  22.0  23.0   0.047   2.0
3   7.0  41.0   0.069   3.0
     method  inversion  cophenetic_corr  top_height  n_inversions
0    single      False            0.759       0.711             0
1  complete      False            0.691       2.798             0
2   average      False            0.805       1.582             0
3  centroid       True            0.821       1.691             1
4    median       True            0.756       1.737             1
5      ward      False            0.630       3.500             0
6    medoid      False            0.810       1.511             0
          single  complete  average  centroid  median  ward  medoid
single      1.00      0.08     0.46      0.54    0.21  0.04    0.54
complete    0.08      1.00     0.12      0.08    0.41  0.80    0.08
average     0.46      0.12     1.00      0.90    0.40  0.18    0.90
centroid    0.54   

## 1.8 Dendrogram cut explorer (Complete linkage on S40)
Complete linkage is monotone here (no inversions) and its height has a plain meaning: the **largest Euclidean distance (log10 m.u.) between any two customers inside the merged cluster**, so a cut at height h guarantees that no two customers in one cluster are farther apart than h. We cut at three heights that sit inside gaps of the sorted merge heights.

In [59]:
Zav = Zs40["complete"]; hts = Zav[:, 2]; print("top 8 merge heights:", np.round(hts[-8:], 3))
CUTS = [2.40, 1.80, 1.40]
states = []
for i_, hcut in enumerate(CUTS):
    lab = fcluster(Zav, hcut, "distance") - 1; k_ = len(set(lab)); lab_c = np.array(canon(lab)); assert canon(lab) == canon(partition_k(Zav, 40, k_))
    order = sorted(range(k_), key=lambda c: -np.sum(lab_c == c)); mp = {c: r for r, c in enumerate(order)}; lab_r = np.array([mp[c] for c in lab_c])
    members = {int(c + 1): S40.source_row_id[lab_r == c].tolist() for c in range(k_)}
    states.append({"cut": hcut, "k": k_, "sizes": np.bincount(lab_r).tolist(), "labels": lab_r, "members": members})
    print(f"cut {hcut}: K={k_} sizes={states[-1]['sizes']}")
    fig, ax = plt.subplots(figsize=(12.5, 5.3))
    thr = hcut
    dendrogram(Zav, labels=S40.source_row_id.values, ax=ax, color_threshold=thr, above_threshold_color=GRID, leaf_rotation=90, leaf_font_size=9, link_color_func=None)
    ax.axhline(hcut, color=ORANGE, lw=2.5); ax.text(ax.get_xlim()[1], hcut, f"  cut = {hcut:.2f}", color=ORANGE, va="bottom", ha="right", fontweight="bold")
    ax.set_ylabel("Complete-linkage height\n(max Euclid. distance, log10 m.u.)"); ax.set_ylim(0, 3.0); ax.set_title(f"Cut at {hcut:.2f} → {k_} clusters"); ax.grid(axis="x", visible=False)
    fig.tight_layout(); save(fig, f"w1_cut_state{i_+1}.png")
    fig, ax = plt.subplots(figsize=(6.2, 5.3))
    for c in range(k_): p_ = XS40[lab_r == c]; ax.scatter(*p_.T, s=54, color=CL[c % 8], marker=MK[c % 8], label=f"cluster {c+1} (n={len(p_)})")
    ax.set_xlabel("log10 Fresh (log10 m.u.)"); ax.set_ylabel("log10 Grocery (log10 m.u.)"); ax.legend(fontsize=10, loc="lower right"); ax.set_xlim(XS40[:, 0].min() - .2, XS40[:, 0].max() + .2); ax.set_ylim(XS40[:, 1].min() - .2, XS40[:, 1].max() + .2)
    ax.set_title(f"{k_} clusters"); fig.tight_layout(); save(fig, f"w1_cut_state{i_+1}_scatter.png")
RES["cut_explorer"] = states; RES["avg_top_heights"] = hts[-8:]

top 8 merge heights: [0.649 0.915 0.92  1.067 1.261 1.607 2.094 2.798]
cut 2.4: K=2 sizes=[37, 3]
saved figs\w1_cut_state1.png
saved figs\w1_cut_state1_scatter.png
cut 1.8: K=3 sizes=[27, 10, 3]
saved figs\w1_cut_state2.png
saved figs\w1_cut_state2_scatter.png
cut 1.4: K=4 sizes=[14, 13, 10, 3]
saved figs\w1_cut_state3.png
saved figs\w1_cut_state3_scatter.png


## 1.9 Interpretation, post-hoc Channel/Region, and a wider check
Cluster **profiles** below are *descriptive*: clusters are fitted on the two log features only; the other four categories, `Channel` and `Region` are looked at afterwards.
Reference partition: Ward, K = 4 on S40 (Ward is the most balanced here; §1.7 shows how much the others agree).

In [60]:
REF = PM["ward"]; S40["ward4"] = REF + 1
prof = S40.groupby("ward4")[CAT].median().round(0).astype(int); prof.insert(0, "n", S40.groupby("ward4").size())
ct_ch = pd.crosstab(S40.ward4, S40.Channel); ct_rg = pd.crosstab(S40.ward4, S40.Region); print(prof); print(ct_ch); print(ct_rg)
RES["profiles_ward4"] = prof.reset_index().to_dict("records"); RES["crosstab_channel"] = ct_ch.reset_index().to_dict("records"); RES["crosstab_region"] = ct_rg.reset_index().to_dict("records")
fig, ax = plt.subplots(figsize=(9.5, 4.6)); sh = np.log10(prof[CAT].values.astype(float)); im = ax.imshow(sh, cmap="Blues", aspect="auto")
ax.set_xticks(range(6), CAT, rotation=20); ax.set_yticks(range(4), [f"cluster {k} (n={n})" for k, n in zip(prof.index, prof.n)]); ax.grid(False)
for i in range(4):
    for j in range(6): ax.text(j, i, f"{prof[CAT].values[i, j]:,}", ha="center", va="center", color="white" if sh[i, j] > sh.mean() else INK, fontsize=11)
ax.set_title("Median annual spending (m.u.) by Ward K=4 cluster — descriptive only"); fig.tight_layout(); save(fig, "w1_profiles.png")
# Ward K=4 scatter with annotated profile
fig, ax = plt.subplots(figsize=(7.2, 5.6))
for k in range(4): p_ = XS40[REF == k]; ax.scatter(*p_.T, s=60, color=CL[k], marker=MK[k], label=f"cluster {k+1} (n={len(p_)})")
ax.set_xlabel("log10 Fresh (log10 m.u.)"); ax.set_ylabel("log10 Grocery (log10 m.u.)"); ax.legend(loc="lower right"); ax.set_title("Ward, K = 4 (S40)"); save(fig, "w1_ward4_scatter.png")

# wider check: same two features, ALL 440 rows (labelled as the larger view)
Xall = w[["lf", "lg"]].values; Zall = linkage(Xall, "ward"); Pall = partition_k(Zall, 440, 4)
ari_wide = adjusted_rand_score(REF, Pall[S40.source_row_id.values - 1])      # row r of the CSV sits at position r-1 in `w`; print("ARI(S40 Ward K=4, restriction of the 440-row Ward K=4) =", round(ari_wide, 3))
wide_prof = w.assign(c=Pall + 1).groupby("c")[CAT].median().round(0).astype(int); wide_prof.insert(0, "n", np.bincount(Pall)); print(wide_prof)
RES["wide_check"] = {"ari": ari_wide, "profiles": wide_prof.reset_index().to_dict("records")}
fig, ax = plt.subplots(figsize=(7.2, 5.6))
for k in range(4): p_ = Xall[Pall == k]; ax.scatter(*p_.T, s=16, color=CL[k], marker=MK[k], alpha=.8, label=f"cluster {k+1} (n={len(p_)})")
ax.scatter(*XS40.T, s=60, facecolor="none", edgecolor=INK, lw=1, label="S40 rows"); ax.set_xlabel("log10 Fresh (log10 m.u.)"); ax.set_ylabel("log10 Grocery (log10 m.u.)"); ax.legend(loc="lower right", fontsize=9); ax.set_title("Wider view: Ward K=4 on all 440 rows"); save(fig, "w1_wide440.png")

        n  Fresh  Milk  Grocery  Frozen  Detergents_Paper  Delicassen
ward4                                                                
1      11   5181  9810    13567    1465              4973        1776
2      13  12759  7330     6964    1646              2730        2503
3       6   1354  3658     2568     470               498         900
4      10  11150  1780     1708    1217               344         570
Channel   1  2
ward4         
1         3  8
2         6  7
3         5  1
4        10  0
Region  1  2  3
ward4          
1       2  2  7
2       3  1  9
3       0  1  5
4       2  0  8
saved figs\w1_profiles.png
saved figs\w1_ward4_scatter.png
     n  Fresh  Milk  Grocery  Frozen  Detergents_Paper  Delicassen
c                                                                 
1  194  11152  1790     2156    2225               312         704
2  128   2822  6141     9386     986              3528        1092
3   88  22631  6744    10285    2242              2384        1860


## 1.10 BIRCH and CHAMELEON — small views on the same S40 data
*BIRCH* stores each sub-cluster as a Clustering Feature **CF = (N, LS, SS)** (count, linear sum, sum of squares). CFs are additive, so a point can be absorbed without keeping the point itself.
*CHAMELEON* builds a k-nearest-neighbour graph, splits it into many small sub-clusters, and merges the pair whose **relative interconnectivity (RI)** and **relative closeness (RC)** are both high. Below: real CF numbers for three S8 customers, a BIRCH fit, and the 3-NN graph (illustration only — scikit-learn has no CHAMELEON).

In [61]:
def CF(P): P = np.atleast_2d(P); return {"N": len(P), "LS": P.sum(0), "SS": float((P ** 2).sum())}
cf_de, cf_a, cf_ade = CF(XS8[[3, 4]]), CF(XS8[[0]]), CF(XS8[[0, 3, 4]])
add = {"N": cf_de["N"] + cf_a["N"], "LS": cf_de["LS"] + cf_a["LS"], "SS": cf_de["SS"] + cf_a["SS"]}
assert add["N"] == cf_ade["N"] and np.allclose(add["LS"], cf_ade["LS"]) and abs(add["SS"] - cf_ade["SS"]) < 1e-12
cen = cf_ade["LS"] / cf_ade["N"]; rad = float(np.sqrt(cf_ade["SS"] / cf_ade["N"] - cen @ cen)); print("CF(a,d,e):", cf_ade, "centroid", cen.round(4), "radius", round(rad, 4))
br = Birch(threshold=0.25, branching_factor=4, n_clusters=None).fit(XS40); print("BIRCH threshold=0.25, B=4 → leaf sub-clusters:", len(br.subcluster_centers_))
nn = NearestNeighbors(n_neighbors=4).fit(XS40); dk, ik = nn.kneighbors(XS40); edges = {(min(i, j), max(i, j)): d for i in range(40) for d, j in zip(dk[i, 1:], ik[i, 1:])}
cross = [(e, d) for e, d in edges.items() if REF[e[0]] != REF[e[1]]]
RES["birch_chameleon"] = {"CF_de": cf_de, "CF_a": cf_a, "CF_ade": cf_ade, "centroid": cen, "radius": rad, "birch_leaves": len(br.subcluster_centers_), "knn_edges": len(edges), "cross_edges": len(cross), "mean_edge": float(np.mean(list(edges.values()))), "mean_cross": float(np.mean([d for _, d in cross]))}
fig, axs = plt.subplots(1, 2, figsize=(13, 5.4))
a = axs[0]; a.scatter(*XS40.T, s=30, color=BLUE); sc = br.subcluster_centers_; lab_b = br.predict(XS40)
for c in range(len(sc)): m_ = XS40[lab_b == c]; a.scatter(*sc[c], marker="s", s=120, facecolor="none", edgecolor=ORANGE, lw=2)
a.set_title(f"BIRCH leaf sub-clusters (squares): {len(sc)} summaries for 40 points"); a.set_xlabel("log10 Fresh"); a.set_ylabel("log10 Grocery")
a = axs[1]
for (i, j), d in edges.items(): a.plot(*XS40[[i, j]].T, color=ORANGE if REF[i] != REF[j] else GRID, lw=2 if REF[i] != REF[j] else 1.2, zorder=1)
for k in range(4): a.scatter(*XS40[REF == k].T, s=46, color=CL[k], marker=MK[k], zorder=3)
a.set_title(f"3-NN graph: {len(cross)} of {len(edges)} edges cross Ward clusters (orange)"); a.set_xlabel("log10 Fresh"); a.set_ylabel("log10 Grocery"); fig.tight_layout(); save(fig, "w1_birch_chameleon.png")

CF(a,d,e): {'N': 3, 'LS': array([11.26391926, 11.97037696]), 'SS': 90.07193914390378} centroid [3.7546 3.9901] radius 0.0745
BIRCH threshold=0.25, B=4 → leaf sub-clusters: 10
saved figs\w1_birch_chameleon.png


# Case 2 — USGS Ridgecrest 2019 high-resolution seismic catalog (Shelly)
**Teaching question.** How can an analyst triage the spatial distribution of recorded events after the mainshock and identify dense concentrations and isolated events for further examination?

*Interpretation limit.* A "cluster" below is a **concentration of recorded events under this 2-D, fixed-ε analysis**. Calling it a fault, a segment or a physical process needs independent geological/seismological evidence.

In [62]:
RAW = DATA / "ridgecrest_shelly2019_catalog_raw.txt"
lines = RAW.read_text().splitlines(); hdr = [l for l in lines if l.startswith("#")]
print("\n".join(hdr[:6]), "\n  ...\n", "\n".join(hdr[8:10]), sep="")
src_line = [i + 1 for i, l in enumerate(lines) if l.strip() and not l.startswith("#")]
ev = pd.read_csv(RAW, comment="#", sep=r"\s+", header=None, names=["year", "month", "day", "hour", "minute", "second", "lat", "lon", "depth_km", "mag", "event_id"])
ev.insert(0, "source_line", src_line)
ev["t_utc"] = pd.to_datetime(dict(year=ev.year, month=ev.month, day=ev.day, hour=ev.hour, minute=ev.minute)) + pd.to_timedelta(ev.second, unit="s")
assert len(ev) == 34091 and ev.event_id.is_unique and ev.t_utc.is_monotonic_increasing
ms = ev.loc[ev.mag.idxmax()]; assert ms.mag == 7.1
print(f"events: {len(ev)} | span {ev.t_utc.min()} → {ev.t_utc.max()} | mainshock M{ms.mag} at {ms.t_utc} (lat {ms.lat:.4f}, lon {ms.lon:.4f}, depth {ms.depth_km} km)")
print(ev[["lat", "lon", "depth_km", "mag"]].describe().round(3).T[["min", "50%", "max"]])
SCHEMA = {"columns": "YYYY MM dd hh mm ss.sss lat lon depth mag ID (whitespace-separated; '#' header lines)", "n_events": len(ev), "header_lines": len(hdr),
          "time_convention": "Header gives no time zone. The M7.1 event is at 2019-07-06 03:19:52.86; the widely published origin time is 03:19:53 UTC, so the times are treated as UTC (empirical check, not stated in the file).",
          "depth": "km, nominally referenced to the surface taken as 0.7 km above sea level (header)", "mag": "SCSN preferred magnitude for SCSN events; calibrated ML-like scaling for template-detected events (header)",
          "ids": "event_id values are not CISN IDs (header)", "caveat": "Events not individually checked; large errors may exist (header)"}
RES["ridgecrest_schema"] = SCHEMA; RES["mainshock"] = {"t": str(ms.t_utc), "lat": ms.lat, "lon": ms.lon, "depth": ms.depth_km, "mag": ms.mag, "event_id": int(ms.event_id)}
RES["ridgecrest_stats"] = {"n": len(ev), "depth_max": float(ev.depth_km.max()), "n_after_mainshock": int((ev.t_utc > ms.t_utc).sum()), "n_before_or_at": int((ev.t_utc <= ms.t_utc).sum())}

# This file contains earthquake hypocentroids for the Ridgecrest foreshock and aftershock sequence
# This version produced 9/4/2019, with data processed for 7/4/2019-7/16/2019 
# (includes foreshock sequence and first ~11 days of M 7.1 aftershocks)  
# Please contact David Shelly (dshelly@usgs.gov) for the most up-to-date version
#
# This relocated dataset is described in detail here:
  ...
# submitted to Seismological Research Letters. 
#
events: 34091 | span 2019-07-04 15:35:29.400000 → 2019-07-16 23:59:26.500000 | mainshock M7.1 at 2019-07-06 03:19:52.860000 (lat 35.7718, lon -117.5942, depth 3.241 km)
              min      50%      max
lat        35.470   35.747   36.036
lon      -117.943 -117.575 -117.213
depth_km    0.089    5.041  556.580
mag        -0.340    1.110    7.100


## 2.1 Coordinates: local azimuthal-equidistant projection in kilometres
Latitude/longitude degrees are not equal-length, so Euclidean ε circles in degrees would be ellipses on the ground. We project to an **azimuthal equidistant** plane centred on the mainshock epicentre
(`+proj=aeqd`, units = km, x = east, y = north). Over this ~70 km area distances are near-exact; we verify against great-circle (haversine) distances. The 2-D analysis uses **epicentres only — depth is ignored**, so events at different depths but the same map position count as neighbours; we quantify that below.

In [63]:
TR = Transformer.from_crs("EPSG:4326", f"+proj=aeqd +lat_0={ms.lat} +lon_0={ms.lon} +datum=WGS84 +units=km", always_xy=True)
ev["x_km"], ev["y_km"] = TR.transform(ev.lon.values, ev.lat.values)
def haversine(la1, lo1, la2, lo2, Rk=6371.0088):
    p1, p2 = np.radians(la1), np.radians(la2); a = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(np.radians(lo2 - lo1) / 2) ** 2; return 2 * Rk * np.arcsin(np.sqrt(a))
rng = np.random.default_rng(0); ia, ib = rng.integers(0, len(ev), 5000), rng.integers(0, len(ev), 5000); ok = ia != ib
dp = np.hypot(ev.x_km.values[ia] - ev.x_km.values[ib], ev.y_km.values[ia] - ev.y_km.values[ib])[ok]; dh = haversine(ev.lat.values[ia], ev.lon.values[ia], ev.lat.values[ib], ev.lon.values[ib])[ok]
rel = np.abs(dp - dh) / np.maximum(dh, 1e-9); sel_ = dh > 5
print(f"projected vs haversine: max |difference| = {np.abs(dp-dh).max():.3f} km; max relative error for pairs > 5 km apart = {rel[sel_].max()*100:.2f} %")
RES["projection"] = {"proj4": f"+proj=aeqd +lat_0={ms.lat:.6f} +lon_0={ms.lon:.6f} +datum=WGS84 +units=km", "max_abs_diff_km": float(np.abs(dp - dh).max()), "max_rel_err_pct_gt5km": float(rel[sel_].max() * 100)}

projected vs haversine: max |difference| = 0.102 km; max relative error for pairs > 5 km apart = 0.23 %


## 2.2 Reproducible selection: analysis set R and the compact hand-trace view H
**R (analysis subset):** all events with `t_utc` **strictly after the mainshock origin time and within the next 24 h**, no magnitude or spatial filter. The catalogue is known to be less complete in the first hours after a large event, so R describes *recorded* events, not all events that occurred.
**H (hand-trace view):** a small square window cut out of R. Rule: scan R in time order; for half-widths 1.5, 2.0, 2.5 km take the first window centred on an R event that holds 18–30 events and, when fitted *alone* with the candidate ε and MinPts, shows ≥3 core points, ≥2 border points, ≥2 noise points and ≥1 provisional-noise point that is later reached. The smallest half-width that passes is used.
**Thinning.** Removing events removes neighbours: we compare R with R_M2 (only M ≥ 2.0).

In [64]:
R = ev[(ev.t_utc > ms.t_utc) & (ev.t_utc <= ms.t_utc + pd.Timedelta(hours=24))].reset_index(drop=True)
XR = R[["x_km", "y_km"]].values; MINPTS = 5; print("R:", len(R), "events | mag range", R.mag.min(), R.mag.max(), "| depth median", R.depth_km.median())
RES["R_info"] = {"n": len(R), "t0": str(R.t_utc.min()), "t1": str(R.t_utc.max()), "mag_min": float(R.mag.min()), "mag_max": float(R.mag.max()), "depth_median": float(R.depth_km.median()), "depth_max": float(R.depth_km.max())}
pad = 2.0; XLIM = (np.floor(XR[:, 0].min() - pad), np.ceil(XR[:, 0].max() + pad)); YLIM = (np.floor(XR[:, 1].min() - pad), np.ceil(XR[:, 1].max() + pad)); print("fixed map extents (km):", XLIM, YLIM)
RES["extents"] = {"x": XLIM, "y": YLIM}

def kdist(X, m=MINPTS):
    """k-distance with the SELF-NEIGHBOUR convention: kneighbors on the training set returns the point itself as neighbour #1 (distance 0), so the last of m columns is the
       distance to the (m-1)-th OTHER point = the m-th point counting itself. p is core under DBSCAN(eps, min_samples=m) <=> kdist(p) <= eps."""
    return NearestNeighbors(n_neighbors=m).fit(X).kneighbors(X)[0][:, -1]
kd = kdist(XR); kd_sorted = np.sort(kd)[::-1]
def knee(a):                                    # farthest point below the chord joining the ends of the ascending curve
    a = np.sort(a); xs = np.arange(len(a)) / (len(a) - 1); ys = (a - a.min()) / (a.max() - a.min()); i = int(np.argmax(xs - ys)); return float(a[i]), i / (len(a) - 1)
knees = {m: knee(kdist(XR, m)) for m in (4, 5, 6, 7, 8)}; print("chord-knee estimate by MinPts:", {m: (round(v[0], 3), round(v[1], 3)) for m, v in knees.items()})
EPS = {"smaller": 0.5, "candidate": 1.0, "larger": 2.0}
db = DBSCAN(eps=EPS["candidate"], min_samples=MINPTS).fit(XR); core_flag = np.zeros(len(XR), bool); core_flag[db.core_sample_indices_] = True
cnt = np.array([len(x) for x in NearestNeighbors(radius=EPS["candidate"]).fit(XR).radius_neighbors(XR, return_distance=False)])       # |N_eps(p)| includes p itself
assert (core_flag == (kd <= EPS["candidate"])).all() and (core_flag == (cnt >= MINPTS)).all()     # k-distance rule == brute-force neighbourhood count == scikit-learn core points
print("k-dist rule, neighbourhood-count rule and sklearn core_sample_indices_ agree for all", len(XR), "events")
RES["kdist"] = {"minpts": MINPTS, "sorted_desc": kd_sorted[::25], "quantiles": {q: float(np.quantile(kd, q)) for q in (.5, .9, .95, .99)}, "knees": {m: v for m, v in knees.items()}, "eps": EPS}

fig, ax = plt.subplots(figsize=(9.5, 5.4))
ax.plot(np.arange(1, len(kd_sorted) + 1), kd_sorted, color=BLUE, lw=2.2); ax.set_yscale("log")
for nm, v in EPS.items(): ax.axhline(v, color=ORANGE if nm == "candidate" else TEAL, ls="-" if nm == "candidate" else "--", lw=2); ax.text(len(kd_sorted) * .98, v * 1.06, f"{nm} ε = {v} km", ha="right", color=ORANGE if nm == "candidate" else TEAL, fontweight="bold", fontsize=11)
ax.axhspan(knees[4][0], knees[8][0], color=ORANGE, alpha=.12); ax.set_xlabel("events, sorted by 5-distance (descending rank)"); ax.set_ylabel("5-distance (km, log scale)")
ax.set_title(f"Sorted k-distance graph, MinPts = {MINPTS} (self counted) · n = {len(XR)}"); save(fig, "r2_kdist.png")

R: 3505 events | mag range 0.59 5.5 | depth median 5.78
fixed map extents (km): (np.float64(-21.0), np.float64(33.0)) (np.float64(-35.0), np.float64(29.0))
chord-knee estimate by MinPts: {4: (0.925, 0.944), 5: (0.981, 0.939), 6: (1.065, 0.938), 7: (1.008, 0.93), 8: (1.137, 0.934)}
k-dist rule, neighbourhood-count rule and sklearn core_sample_indices_ agree for all 3505 events
saved figs\r2_kdist.png


In [65]:
def dbscan_trace(X, eps, m):
    """Textbook DBSCAN with an explicit FIFO queue, logging every step. Points are processed in index (= time) order."""
    n = len(X); Dm = np.sqrt(((X[:, None] - X[None]) ** 2).sum(-1)); N = [np.where(Dm[i] <= eps)[0] for i in range(n)]   # N_eps(i) includes i
    core = np.array([len(a) >= m for a in N]); UNV, NOI = -9, -1; lab = np.full(n, UNV); log = []; cid = -1; rescued = []; queue = []
    def snap(kind, p, **kw): log.append({"kind": kind, "point": int(p), "labels": lab.tolist(), "queue": [int(q) for q in queue], "cluster": cid, **kw})
    for i in range(n):
        if lab[i] != UNV: continue
        snap("query", i, n_eps=len(N[i]))
        if not core[i]: lab[i] = NOI; snap("provisional_noise", i); continue
        cid += 1; lab[i] = cid; queue = [int(j) for j in N[i] if j != i]; snap("seed", i)
        while queue:
            j = queue.pop(0)
            if lab[j] == NOI: lab[j] = cid; rescued.append(int(j)); snap("noise_reached_becomes_border", j); continue
            if lab[j] != UNV: continue
            lab[j] = cid; snap("query", j, n_eps=len(N[j]))
            if core[j]:
                queue.extend(int(k) for k in N[j] if lab[k] in (UNV, NOI) and int(k) not in queue); snap("core_expands", j)
            else: snap("border_absorbed", j)
    return lab, core, N, log, rescued, Dm

HM = None
for hw in (1.5, 2.0, 2.5):
    for ci in range(len(XR)):
        idx = np.where((np.abs(XR[:, 0] - XR[ci, 0]) <= hw) & (np.abs(XR[:, 1] - XR[ci, 1]) <= hw))[0]
        if not 18 <= len(idx) <= 30: continue
        lab_h, core_h, _N, _log, resc, _D = dbscan_trace(XR[idx], EPS["candidate"], MINPTS)
        nb = int(((lab_h >= 0) & ~core_h).sum()); nn_ = int((lab_h == -1).sum())
        if core_h.sum() >= 3 and nb >= 2 and nn_ >= 2 and len(resc) >= 1: HM = (hw, ci, idx); break
    if HM: break
hw, ci, hidx = HM; print(f"H window: half-width {hw} km around R event index {ci} (event_id {R.event_id[ci]}) → n = {len(hidx)}")
Hdf = R.iloc[hidx].copy(); Hdf.insert(0, "pt", [f"e{k+1:02d}" for k in range(len(Hdf))]); XH = Hdf[["x_km", "y_km"]].values
labH, coreH, NH, logH, rescH, DH = dbscan_trace(XH, EPS["candidate"], MINPTS)
sk_h = DBSCAN(eps=EPS["candidate"], min_samples=MINPTS).fit(XH); core_sk = np.zeros(len(XH), bool); core_sk[sk_h.core_sample_indices_] = True
assert (coreH == core_sk).all() and ((labH == -1) == (sk_h.labels_ == -1)).all() and len(set(labH[labH >= 0])) == len(set(sk_h.labels_[sk_h.labels_ >= 0]))
assert adjusted_rand_score(labH[coreH], sk_h.labels_[coreH]) == 1.0
print("hand-coded trace == scikit-learn on H: same core set, same noise set, same core partition")
stat = lambda lab, core: np.where(lab == -1, "noise", np.where(core, "core", "border"))
statH = stat(labH, coreH); print(pd.Series(statH).value_counts().to_dict(), "| clusters", len(set(labH[labH >= 0])), "| provisional noise later reached:", [Hdf.pt.iloc[k] for k in rescH])

# Does the small view reproduce the larger fit?  Compare each H event's status in the H-only fit with its status in the R fit at the same eps / MinPts.
labR_c = db.labels_; statR = stat(labR_c, core_flag)[hidx]
same = statH == statR; print(f"status agreement H-only fit vs R fit: {same.sum()}/{len(hidx)}"); print(pd.crosstab(pd.Series(statH, name="H-only fit"), pd.Series(statR, name="R fit")))
nbr_loss = np.array([len(NH[i]) for i in range(len(hidx))]); nbr_R = cnt[hidx]
print("neighbour counts |N_eps| (incl. self): H-only vs R →", list(zip(nbr_loss.tolist(), nbr_R.tolist())))
RES["H"] = {"half_width": hw, "center_event_id": int(R.event_id[ci]), "n": len(hidx), "events": Hdf[["pt", "event_id", "source_line", "x_km", "y_km", "mag", "depth_km"]].to_dict("records"),
            "status_H": statH, "status_R": statR, "labels_H": labH, "n_eps_H": nbr_loss, "n_eps_R": nbr_R, "agree": int(same.sum()), "rescued": [Hdf.pt.iloc[k] for k in rescH],
            "log": [{k: v for k, v in e.items()} for e in logH], "neighbors": [a.tolist() for a in NH], "window_center_xy": XR[ci], "eps": EPS["candidate"], "minpts": MINPTS}
prov_h = ["Source: USGS data release, Shelly (2020) Ridgecrest catalog, DOI 10.5066/P9JN6H0N (CC0); file ridgecrest_shelly2019_catalog_raw.txt (unchanged copy).",
          "event_id and source_line identify each row in that file; x_km,y_km = azimuthal-equidistant projection centred on the M7.1 epicentre (km).",
          f"Selection R: events after the M7.1 origin time within 24 h. H: first square window (half-width {hw} km, centred on event_id {int(R.event_id[ci])}) that meets the stated trace criteria."]
H_out = Hdf.assign(status_H_only_fit=statH, status_R_fit=statR, cluster_H_only_fit=labH)[["pt", "event_id", "source_line", "t_utc", "lat", "lon", "depth_km", "mag", "x_km", "y_km", "status_H_only_fit", "status_R_fit", "cluster_H_only_fit"]]
prov_csv(H_out, DATA / "ridgecrest_H_hand_trace_events.csv", prov_h)
prov_csv(R[["event_id", "source_line", "t_utc", "lat", "lon", "depth_km", "mag", "x_km", "y_km"]], DATA / "ridgecrest_R_first24h_after_mainshock.csv", prov_h[:2] + [prov_h[2]])

fig, ax = plt.subplots(figsize=(6.6, 6.0)); ax.set_aspect("equal")
for nm, mk, col in [("core", "o", BLUE), ("border", "^", ORANGE), ("noise", "x", "#64748B")]:
    m_ = statH == nm; ax.scatter(*XH[m_].T, marker=mk, s=70, color=col, label=f"{nm} (n={m_.sum()})")
for k, (x_, y_) in enumerate(XH): ax.add_patch(plt.Circle((x_, y_), EPS["candidate"], fill=False, ec=GRID, lw=.8)); ax.annotate(Hdf.pt.iloc[k][1:], (x_, y_), xytext=(4, 4), textcoords="offset points", fontsize=8)
ax.set_xlabel("x east of mainshock (km)"); ax.set_ylabel("y north of mainshock (km)"); ax.legend(); ax.set_title(f"Hand-trace view H (n={len(XH)}), ε={EPS['candidate']} km, MinPts={MINPTS}"); save(fig, "r2_H_status.png")

H window: half-width 1.5 km around R event index 1589 (event_id 70649469) → n = 19
hand-coded trace == scikit-learn on H: same core set, same noise set, same core partition
{'core': 13, 'border': 4, 'noise': 2} | clusters 2 | provisional noise later reached: ['e03']
status agreement H-only fit vs R fit: 16/19
R fit       border  core  noise
H-only fit                     
border           2     2      0
core             0    13      0
noise            1     0      1
neighbour counts |N_eps| (incl. self): H-only vs R → [(11, 33), (11, 24), (4, 7), (11, 34), (2, 3), (9, 17), (5, 5), (3, 3), (9, 36), (4, 4), (3, 3), (11, 37), (6, 6), (4, 11), (10, 15), (9, 35), (11, 16), (11, 34), (11, 38)]
saved figs\r2_H_status.png


## 2.3 Three ε states on the SAME events, projection, extents and MinPts
`DBSCAN(eps, min_samples=5)` on R. `core_sample_indices_` → core; labelled but not core → border; label −1 → noise. For the same ε we also record how many neighbour entries scikit-learn has to materialise.

In [66]:
FITS = {}
for nm, eps in EPS.items():
    d_ = DBSCAN(eps=eps, min_samples=MINPTS).fit(XR); lab = d_.labels_; core = np.zeros(len(XR), bool); core[d_.core_sample_indices_] = True
    ordr = np.argsort(-np.bincount(lab[lab >= 0])) if (lab >= 0).any() else []; remap = {int(o): r for r, o in enumerate(ordr)}; lab_r = np.array([remap[l] if l >= 0 else -1 for l in lab])
    pairs = int(sum(len(a) for a in NearestNeighbors(radius=eps).fit(XR).radius_neighbors(XR, return_distance=False)))
    sizes_ = np.bincount(lab_r[lab_r >= 0]).tolist()
    FITS[nm] = {"eps": eps, "labels": lab_r, "core": core, "n_clusters": len(sizes_), "n_core": int(core.sum()), "n_border": int(((lab_r >= 0) & ~core).sum()), "n_noise": int((lab_r == -1).sum()), "sizes": sizes_, "neighbor_entries": pairs}
    print(nm, {k: v for k, v in FITS[nm].items() if k not in ("labels", "core", "sizes")}, "largest clusters:", sizes_[:5])
    assert FITS[nm]["n_core"] + FITS[nm]["n_border"] + FITS[nm]["n_noise"] == len(XR)
RES["eps_states"] = {nm: {k: v for k, v in f.items() if k not in ("core",)} | {"core_count": f["n_core"]} for nm, f in FITS.items()}
def eps_map(nm, f):
    lab, core = f["labels"], f["core"]; fig, ax = plt.subplots(figsize=(7.4, 6.9)); ax.set_aspect("equal")
    m_ = lab == -1; ax.scatter(*XR[m_].T, marker="x", s=14, color="#64748B", lw=.9, label=f"noise (n={f['n_noise']})")
    for c in range(f["n_clusters"]):
        mc = lab == c; col = CL[c % 8]
        ax.scatter(*XR[mc & core].T, marker="o", s=9, color=col, lw=0, label="core" if c == 0 else None); ax.scatter(*XR[mc & ~core].T, marker="^", s=22, color=col, edgecolor=INK, lw=.4, label="border" if c == 0 else None)
    ax.scatter(0, 0, marker="*", s=240, color=INK, zorder=6, label="M7.1 epicentre"); ax.set_xlim(*XLIM); ax.set_ylim(*YLIM); ax.set_xlabel("x east of mainshock (km)"); ax.set_ylabel("y north of mainshock (km)")
    ax.set_title(f"ε = {f['eps']} km · MinPts = {MINPTS} → {f['n_clusters']} clusters"); h, l = ax.get_legend_handles_labels()
    h = [Line2D([], [], marker="o", color="w", markerfacecolor=BLUE, markersize=7), Line2D([], [], marker="^", color="w", markerfacecolor=BLUE, markeredgecolor=INK, markersize=8), h[0], h[-1]]
    ax.legend(h, ["core (colour = cluster)", "border (colour = cluster)", f"noise (n={f['n_noise']})", "M7.1 epicentre"], loc="lower left", fontsize=9, framealpha=.9)
    save(fig, f"r2_eps_{nm}.png")
for nm, f in FITS.items(): eps_map(nm, f)
st_out = R[["event_id", "source_line", "x_km", "y_km", "mag"]].copy()
for nm, f in FITS.items(): st_out[f"cluster_eps_{f['eps']}"] = f["labels"]; st_out[f"status_eps_{f['eps']}"] = stat(f["labels"], f["core"])
prov_csv(st_out, DATA / "ridgecrest_R_dbscan_three_eps_states.csv", prov_h[:2] + [f"DBSCAN(eps in {list(EPS.values())} km, min_samples={MINPTS}) on R. cluster -1 = noise; clusters ranked by size (0 = largest)."])
fig, axs = plt.subplots(1, 3, figsize=(16, 5.6), sharex=True, sharey=True)
for a, (nm, f) in zip(axs, FITS.items()):
    a.set_aspect("equal"); a.scatter(*XR[f["labels"] == -1].T, marker="x", s=8, color="#64748B", lw=.7)
    for c in range(f["n_clusters"]): a.scatter(*XR[f["labels"] == c].T, s=5, color=CL[c % 8])
    a.set_title(f"ε = {f['eps']} km: {f['n_clusters']} clusters, {f['n_noise']} noise"); a.set_xlim(*XLIM); a.set_ylim(*YLIM)
fig.tight_layout(); save(fig, "r2_eps_triptych.png")

smaller {'eps': 0.5, 'n_clusters': 31, 'n_core': 3001, 'n_border': 146, 'n_noise': 358, 'neighbor_entries': 87265} largest clusters: [2126, 226, 223, 177, 61]
candidate {'eps': 1.0, 'n_clusters': 5, 'n_core': 3292, 'n_border': 40, 'n_noise': 173, 'neighbor_entries': 279233} largest clusters: [3297, 13, 12, 5, 5]
larger {'eps': 2.0, 'n_clusters': 2, 'n_core': 3416, 'n_border': 22, 'n_noise': 67, 'neighbor_entries': 840927} largest clusters: [3432, 6]
saved figs\r2_eps_smaller.png
saved figs\r2_eps_candidate.png
saved figs\r2_eps_larger.png
saved figs\r2_eps_triptych.png


## 2.4 Filtering / thinning, depth, traversal order, and cost
* **Thinning:** keep only M ≥ 2.0 events (R_M2) and refit with the same ε and MinPts — neighbours disappear, so apparent density changes.
* **Depth:** compare the 2-D neighbour counts with a 3-D version (x, y, depth in km).
* **Traversal order:** border points reachable from two clusters are assigned to whichever cluster reaches them first; we refit with the rows reversed and compare.
* **Cost:** wall-clock time and number of materialised neighbour entries (hardware-dependent; read the trend, not the milliseconds).

In [67]:
R2 = R[R.mag >= 2.0].reset_index(drop=True); X2 = R2[["x_km", "y_km"]].values; thin = {}
for nm, eps in EPS.items():
    d2 = DBSCAN(eps=eps, min_samples=MINPTS).fit(X2); c2 = np.zeros(len(X2), bool); c2[d2.core_sample_indices_] = True
    full_core_on_R2 = FITS[nm]["core"][(R.mag >= 2.0).values]
    thin[nm] = {"n": len(X2), "clusters": len(set(d2.labels_[d2.labels_ >= 0])), "noise": int((d2.labels_ == -1).sum()), "noise_frac": float((d2.labels_ == -1).mean()), "core_frac": float(c2.mean()),
                "full_R_noise_frac": float((FITS[nm]["labels"] == -1).mean()), "core_lost_vs_full": int((full_core_on_R2 & ~c2).sum()), "median_5dist": float(np.median(kdist(X2)))}
print(pd.DataFrame(thin).T.round(3)); RES["thinning"] = thin; RES["thinning"]["median_5dist_R"] = float(np.median(kd))
fig, axs = plt.subplots(1, 2, figsize=(12, 4.8))
for lab_, X_, col in [("R: all events (n=%d)" % len(XR), XR, BLUE), ("R_M2: M ≥ 2.0 (n=%d)" % len(X2), X2, ORANGE)]:
    k_ = np.sort(kdist(X_)); axs[0].plot(k_, np.arange(1, len(k_) + 1) / len(k_), color=col, lw=2, label=lab_)
axs[0].set_xscale("log"); axs[0].set_xlabel("5-distance (km)"); axs[0].set_ylabel("fraction of events"); axs[0].legend(); axs[0].set_title("Thinning moves the whole k-distance curve right")
xx = np.arange(3); axs[1].bar(xx - .18, [thin[n]["full_R_noise_frac"] * 100 for n in EPS], .36, color=BLUE, label="R"); axs[1].bar(xx + .18, [thin[n]["noise_frac"] * 100 for n in EPS], .36, color=ORANGE, label="R_M2")
axs[1].set_xticks(xx, [f"ε={v} km" for v in EPS.values()]); axs[1].set_ylabel("noise (% of events)"); axs[1].legend(); axs[1].set_title("Same ε, fewer neighbours → more noise"); fig.tight_layout(); save(fig, "r2_thinning.png")

X3 = np.c_[XR, R.depth_km.values]; cnt3 = np.array([len(a) for a in NearestNeighbors(radius=EPS["candidate"]).fit(X3).radius_neighbors(X3, return_distance=False)])
depth = {"core_2d": int((cnt >= MINPTS).sum()), "core_3d": int((cnt3 >= MINPTS).sum()), "median_neighbors_2d": float(np.median(cnt)), "median_neighbors_3d": float(np.median(cnt3)), "mean_ratio_3d_to_2d": float(np.mean(cnt3 / cnt))}
print("depth treatment at candidate eps:", depth); RES["depth_check"] = depth

shared = {}
for nm, eps in EPS.items():
    f = FITS[nm]; nbr = NearestNeighbors(radius=eps).fit(XR).radius_neighbors(XR, return_distance=False); border = np.where((f["labels"] >= 0) & ~f["core"])[0]
    multi = [i for i in border if len({f["labels"][j] for j in nbr[i] if f["core"][j]}) > 1]
    rev = np.arange(len(XR))[::-1]; dr = DBSCAN(eps=eps, min_samples=MINPTS).fit(XR[rev]); lr = np.empty(len(XR), int); lr[rev] = dr.labels_
    cores = np.where(f["core"])[0]; Cm = pd.crosstab(f["labels"][cores], lr[cores]); mapping = Cm.idxmax(axis=1).to_dict()
    changed = [i for i in multi if mapping.get(f["labels"][i], -2) != lr[i]]
    shared[nm] = {"border_points": int(len(border)), "border_reachable_from_2_clusters": len(multi), "assignment_changed_when_order_reversed": len(changed)}
print(shared); RES["order_dependence"] = shared

def timeit(X, alg, eps=EPS["candidate"], reps=1):
    ts = []
    for _ in range(reps): t0 = time.perf_counter(); DBSCAN(eps=eps, min_samples=MINPTS, algorithm=alg).fit(X); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))
post = ev[ev.t_utc > ms.t_utc]; sets_t = {"R (24 h)": R, "first 72 h": post[post.t_utc <= ms.t_utc + pd.Timedelta(hours=72)], "all post-mainshock": post}
cost = []
for nm, d_ in sets_t.items():
    X_ = d_[["x_km", "y_km"]].values; cost.append({"set": nm, "n": len(X_), "brute_s": timeit(X_, "brute"), "kd_tree_s": timeit(X_, "kd_tree"), "ball_tree_s": timeit(X_, "ball_tree"),
        "neighbor_entries": int(sum(len(a) for a in NearestNeighbors(radius=EPS["candidate"]).fit(X_).radius_neighbors(X_, return_distance=False)))})
print(pd.DataFrame(cost).round(3)); RES["cost"] = cost

                n  clusters  noise  noise_frac  core_frac  full_R_noise_frac  \
smaller    1890.0      33.0  352.0       0.186      0.729              0.102   
candidate  1890.0       3.0  136.0       0.072      0.913              0.049   
larger     1890.0       6.0   47.0       0.025      0.960              0.019   

           core_lost_vs_full  median_5dist  
smaller                211.0         0.322  
candidate               31.0         0.322  
larger                  22.0         0.322  
saved figs\r2_thinning.png
depth treatment at candidate eps: {'core_2d': 3292, 'core_3d': 2741, 'median_neighbors_2d': 46.0, 'median_neighbors_3d': 13.0, 'mean_ratio_3d_to_2d': 0.3309088743686208}
{'smaller': {'border_points': 146, 'border_reachable_from_2_clusters': 6, 'assignment_changed_when_order_reversed': 2}, 'candidate': {'border_points': 40, 'border_reachable_from_2_clusters': 0, 'assignment_changed_when_order_reversed': 0}, 'larger': {'border_points': 22, 'border_reachable_from_2_clust

## 2.5 Fixed-ε limits → OPTICS and HDBSCAN (brief connection)
DBSCAN applies one ε everywhere. OPTICS orders points by *reachability distance* so every ε′ ≤ max_eps can be read from one plot; HDBSCAN (Campello et al.) turns the same idea into a hierarchy and extracts stable clusters without a single global ε.

In [68]:
op = OPTICS(min_samples=MINPTS, max_eps=EPS["larger"], cluster_method="xi", xi=0.05, min_cluster_size=20).fit(XR)
reach = op.reachability_[op.ordering_]; reach_f = np.where(np.isfinite(reach), reach, np.nan)
hd = HDBSCAN(min_cluster_size=20, min_samples=MINPTS, copy=True).fit(XR)
RES["optics_hdbscan"] = {"optics_clusters": int(len(set(op.labels_[op.labels_ >= 0]))), "optics_noise": int((op.labels_ == -1).sum()), "hdbscan_clusters": int(len(set(hd.labels_[hd.labels_ >= 0]))), "hdbscan_noise": int((hd.labels_ == -1).sum()),
                         "params": {"optics": "min_samples=5, max_eps=2.0 km, xi=0.05, min_cluster_size=20", "hdbscan": "min_cluster_size=20, min_samples=5"}}
print(RES["optics_hdbscan"])
fig, axs = plt.subplots(1, 2, figsize=(14, 5.2), gridspec_kw={"width_ratios": [1.25, 1]})
axs[0].fill_between(np.arange(len(reach_f)), np.nan_to_num(reach_f, nan=EPS["larger"]), color=BLUE, step="mid", alpha=.85); axs[0].set_ylim(0, 2.1)
for nm, v in EPS.items(): axs[0].axhline(v, color=ORANGE if nm == "candidate" else TEAL, ls="-" if nm == "candidate" else "--", lw=1.8)
axs[0].set_xlabel("events in OPTICS order"); axs[0].set_ylabel("reachability distance (km)"); axs[0].set_title("OPTICS reachability plot (R, MinPts = 5, max ε = 2 km)")
a = axs[1]; a.set_aspect("equal"); a.scatter(*XR[hd.labels_ == -1].T, marker="x", s=8, color="#64748B", lw=.7)
for c in range(hd.labels_.max() + 1): a.scatter(*XR[hd.labels_ == c].T, s=5, color=CL[c % 8])
a.set_xlim(*XLIM); a.set_ylim(*YLIM); a.set_title(f"HDBSCAN: {RES['optics_hdbscan']['hdbscan_clusters']} clusters, {RES['optics_hdbscan']['hdbscan_noise']} noise"); a.set_xlabel("x east (km)"); a.set_ylabel("y north (km)")
fig.tight_layout(); save(fig, "r2_optics_hdbscan.png")

{'optics_clusters': 48, 'optics_noise': 2149, 'hdbscan_clusters': 36, 'hdbscan_noise': 1286, 'params': {'optics': 'min_samples=5, max_eps=2.0 km, xi=0.05, min_cluster_size=20', 'hdbscan': 'min_cluster_size=20, min_samples=5'}}
saved figs\r2_optics_hdbscan.png


## 3. Export and verification summary
`results/results.json` is the single source for the numbers placed on the slides.

In [69]:
(OUT / "results.json").write_text(json.dumps(jsonable(RES), indent=1))
print("results.json written:", (OUT / "results.json").stat().st_size, "bytes")
print("Wholesale S8 ids:", S8.source_row_id.tolist()); print("S40 n =", len(S40), "| Ridgecrest R n =", len(R), "| H n =", len(hidx))
print({nm: (f["eps"], f["n_clusters"], f["n_core"], f["n_border"], f["n_noise"]) for nm, f in FITS.items()})
print("All assertions passed.")

results.json written: 168147 bytes
Wholesale S8 ids: [2, 57, 112, 167, 222, 277, 332, 387]
S40 n = 40 | Ridgecrest R n = 3505 | H n = 19
{'smaller': (0.5, 31, 3001, 146, 358), 'candidate': (1.0, 5, 3292, 40, 173), 'larger': (2.0, 2, 3416, 22, 67)}
All assertions passed.
